In [ ]:
# Instalamos las dependencias necesarias
!pip install bw2calc -q # Paquete de brightway
!pip install bw2data -q # Paquete de brightway
!pip install bw2io -q # Paquete de brightway
!pip install bw2analyzer # Paquete de brightway
!pip install pandas -q
!pip install pypardiso -q
!pip install mermaid-py -q # Este paquete permite construir diagramas. Lo usaré para el reporte.
!pip install seaborn -q

Debemos descargar un archivo de respaldo que contiene BAFU. Para ello tenemos que autenticar nuestro usuario de **gmail** que fue creado anteriormente.

In [ ]:
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
drive.CreateFile({"id": "1t3qNmOuOgCvT9B6dEmnJVzEspCHspDAT"}).GetContentFile(
    "backup.tar.gz"
)

Verificamos

In [ ]:
!du -hs backup.tar.gz

In [ ]:
import bw2calc as bc
import bw2data as bd
import bw2io as bi
import pandas as pd
import bw2analyzer as ba
from rich import print

### Importar el backup del proyecto
Esta modalidad no requiere mucha explicación: El proyecto se carga nuevamente.

In [ ]:
bi.restore_project_directory(
    "backup.tar.gz",  # nombre del archivo, creado celdas arriba
    project_name="proyecto_bafu",  # Se puede elegir un nombre nuevo para el proyecto
    overwrite_existing=True,
)

In [ ]:
# Seleccionamos el proyecto correspondiente y asignamos las bases de datos a cada variable
bd.projects.set_current("proyecto_bafu")
bafu_db = bd.Database("bafu")
# Set the current project
bio = bd.Database("biosphere3")

### Explorando BAFU

In [ ]:
bd.projects.set_current("proyecto_bafu")
seleccionado = bafu_db.random() # Explora las actividades
print("Proceso aleatorio seleccionado: ",seleccionado)
# Primero exploramos los keys de la actividad seleccionada
print(list(seleccionado.keys()))

In [ ]:
# Podemos ver el contenido de todo el dataset.
print(seleccionado.as_dict())

Como pueden notar, el contenido de la actividad BAFU es bastante rica. Existen campos fuera de `name`, `code`,`location` y `unit` que son nuevos para nosotros, lo que demuestra que brightway es lo suficientemente flexible al definir una actividad.

Lo que vimos en la celda anterior describe a una actividad, pero aun no describe sus conexiones (`exchanges`). Para acceder a ellas, hay que utilizar las funciones `exchanges`, `technosphere` o `biosphere`, segun lo que se desee observar.

In [ ]:
# `exchanges` retorna un objeto the brightway que no es nativo de python
type(seleccionado.exchanges())

In [ ]:
# Si deseamos leerlo al estilo de una lista, hay que convertirlo en una lista.
for exchange in seleccionado.exchanges():
    print(exchange)
# print(list(seleccionado.exchanges()))

In [ ]:
# Si deseamos solo la tecnosfera, usamos la funcion correspondiente
for exchange in seleccionado.technosphere():
    print(exchange)

La impresion realizada en la celda de arriba nos muestra la informacion necesaria para poder construir las matrices. Sin embargo, brightway nos permite manipular el `exchange` y acceder a su metadata.

In [ ]:
# Seleccionamos el segundo `exchange`de la list
# a
exchange = list(seleccionado.technosphere())[1]
print(exchange.as_dict())

## Opciones de busqueda
Como podran imaginar, manipular una base de datos con tantas actividades (~11k) es bastante complicado. Podemos utilizar funciones nativas de python para realizar una busqueda.

In [ ]:
for x in bafu_db:
  if 'Bicycle' in x['name']:
    print(x)

In [ ]:
# Para quienes esten comodos con python, pueden utilizar list-comprehension, and select the
bicycle = [x for x in bafu_db if 'Bicycle' in x['name']]
print(bicycle)
# truck

Esta manera de buscar es mas 'pythonic'. Sin embargo, tambien puedes usar el buscador de brightway a traves de la funcion `search`.

In [ ]:
bafu_db.search('Bicycle')

# Introduccion a brightway - Under the hood

En esta seccion hablaremos de los conceptos fundamentales de brigthway. Es importante aclarar que toda esta informacion esta disponible en linea en la pagina de documentacion:

https://docs.brightway.dev/en/latest/index.html

## Explorando las matrices
Ahora que sabemos como crear un actividad y metodos desde cero. Podemos concentrarnos en manipular las actividades que estan presentes en BAFU.
Para esta parte usaremos un proyecto que hemos preparado para ustedes que contiene una biosfera y tecnosfera compatible con BAFU v2

In [ ]:
# Re-import for this section if running independently
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from rich import print

Tenemos dos bases de datos

In [ ]:
bd.databases

In [ ]:
# seleccionamos la base de datos bafu y una actividad que tomaremos de ejemplo
bicycle = bafu_db.search("Bicycle, conventional, urban, 2020")[-1]
bicycle

In [ ]:
# Elegimos un metodo que ya esta instalado y hace un LCA pero nos detenemos en la etapa de LCI
method=('IPCC 2021', 'climate change', 'global warming potential (GWP100)')
lca = bc.LCA({bicycle:1},method=method) # Instancia la clase
lca.lci() # calcula el inventario de ciclo de vida

La base de datos BAFU contiene actividades de la base de datos suiza
Entonces que dimensiones deberia tener la matriz de la tecnosfera?

In [ ]:
lca.technosphere_matrix.toarray().shape

Que dimensiones deberia tener el vector s?

In [ ]:
lca.supply_array

Si quisiera saber cuanto de otra actividad específica
se requiere en TOTAL para producir la unidad funcional...

In [ ]:
# Ejemplo: buscar una actividad relacionada en la base de datos
otra_act = bafu_db.search('steel')[0]
otra_act

In [ ]:
# el lca.activity_dict me permite ubicar una actividad en la matriz.
lca.supply_array[lca.activity_dict[otra_act.id]]

Ahora continuamos con el LCIA

In [ ]:
lca.lcia() # Calcula los impactos
print("El impacto es: ", lca.score)

## Analisis de contribuciones
Para entender las distintas contribuciones, tenemos que seguir utilizando el objeto LCA.
Este objeto mantiene los resultados del ACV en memoria

### Procesos mas importantes
Para listar los procesos que generan mas impactos utilizaremos el paquete `bw2analyzer` y `pandas`.

In [ ]:
ba.ContributionAnalysis().annotated_top_processes(lca=lca) # dificil de visulizar
ba.ContributionAnalysis.annotated_top_processes??

In [ ]:
pd.DataFrame(
    [(x, y, z["name"]) for x, y, z in ba.ContributionAnalysis().annotated_top_processes(lca=lca)],
    columns=["score", "quantity", "name"]
)

### Emisiones mas importantes
De manera similar, podemos obtener el ranking de flujos ambiental que generan mayores impactos

In [ ]:
import pandas as pd
import bw2analyzer as ba
pd.DataFrame(
    [(x, y, z["name"]) for x, y, z in ba.ContributionAnalysis().annotated_top_emissions(lca=lca)],
    columns=["score", "quantity", "name"]
)

La importancia de las emisiones en el impacto tiene que ver con la cantidad y con los factores de caracterizacion.
Podemos listar estos factores para revisarlos

In [ ]:
for key, cf in bd.Method(method).load()[0:20]:
    # print(key, cf)
    print(bd.Database('biosphere3').get(key[1]), cf)



Pueden revisar
La importancia de las emisiones en el impacto tiene que ver con la cantidad y con los factores de caracterizacion.
Podemos listar estos factores para revisarlos

In [ ]:
ba.print_recursive_calculation(bicycle,lcia_method=method)

In [ ]:
calculations = ba.utils.recursive_calculation_to_object(bicycle,lcia_method=method)

In [ ]:
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from matplotlib.sankey import Sankey


def plot_sankey_tree(
    data: List[Dict[str, Any]],
    title: str = "LCA Contribution Tree",
    value_field: str = "fraction",
    height: int = 800,
    width: int = 1200,
    use_plotly: bool = True,
    max_label_length: int = 50,
    min_fraction: float = 0.0,
    show_values: bool = True,
) -> None:
    """
    Plot a Sankey diagram showing the hierarchical tree structure of LCA contributions.

    The thickness of arrows represents the contribution (fraction) of each node to its parent.

    Parameters
    ----------
    data : List[Dict[str, Any]]
        List of dictionaries, each representing a node in the tree with keys:
        - 'label': unique identifier for the node
        - 'parent': label of parent node (None for root)
        - 'score': impact score
        - 'fraction': contribution to total impact (used for arrow thickness)
        - 'name': descriptive name of the activity
        - 'amount': amount value
        - 'key': unique key tuple
    title : str, optional
        Title of the diagram (default: "LCA Contribution Tree")
    value_field : str, optional
        Field to use for link values ('fraction' or 'score') (default: "fraction")
    height : int, optional
        Height of the figure in pixels (default: 800)
    width : int, optional
        Width of the figure in pixels (default: 1200)
    use_plotly : bool, optional
        If True, use Plotly; if False, use matplotlib (default: True)
    max_label_length : int, optional
        Maximum length for node labels (default: 50)
    min_fraction : float, optional
        Minimum fraction to display (filter small contributions) (default: 0.0)
    show_values : bool, optional
        Whether to show values in labels (default: True)

    Returns
    -------
    None
        Displays the Sankey diagram

    Examples
    --------
    >>> data = [
    ...     {'label': 'root', 'parent': None, 'score': 62.29, 'fraction': 1.0,
    ...      'amount': 1.0, 'name': 'Bicycle, conventional, urban, 2020',
    ...      'key': ('bafu', '522701.0')},
    ...     {'label': 'root_a', 'parent': 'root', 'score': 56.58, 'fraction': 0.908,
    ...      'amount': 0.706, 'name': 'Bicycle, at regional storage',
    ...      'key': ('bafu', '291950.0')},
    ... ]
    >>> plot_sankey_tree(data)
    """

    if use_plotly:
        _plot_sankey_plotly(
            data,
            title,
            value_field,
            height,
            width,
            max_label_length,
            min_fraction,
            show_values,
        )
    else:
        _plot_sankey_matplotlib(
            data,
            title,
            value_field,
            height,
            width,
            max_label_length,
            min_fraction,
            show_values,
        )


def _plot_sankey_plotly(
    data: List[Dict[str, Any]],
    title: str,
    value_field: str,
    height: int,
    width: int,
    max_label_length: int,
    min_fraction: float,
    show_values: bool,
) -> None:
    """Internal function to plot Sankey diagram using Plotly."""

    # Filter data by minimum fraction
    filtered_data = [node for node in data if node.get("fraction", 0) >= min_fraction]

    if not filtered_data:
        print("No data to display after filtering.")
        return

    # Create a mapping from label to index
    label_to_idx = {node["label"]: idx for idx, node in enumerate(filtered_data)}

    # Prepare node labels
    node_labels = []
    for node in filtered_data:
        name = node["name"]
        if len(name) > max_label_length:
            name = name[: max_label_length - 3] + "..."

        if show_values:
            label = f"{name}<br>({node['fraction'] * 100:.1f}%)"
        else:
            label = name

        node_labels.append(label)

    # Prepare links (edges from parent to child)
    sources = []
    targets = []
    values = []
    link_labels = []

    for node in filtered_data:
        if node["parent"] is not None:
            # Check if parent exists in filtered data
            if node["parent"] in label_to_idx:
                parent_idx = label_to_idx[node["parent"]]
                child_idx = label_to_idx[node["label"]]

                sources.append(parent_idx)
                targets.append(child_idx)

                # Use specified field for link value
                value = node.get(value_field, node.get("fraction", 0))
                values.append(value)

                # Create hover text for link
                link_label = f"{node['name'][:30]}<br>Fraction: {node['fraction'] * 100:.2f}%<br>Score: {node['score']:.2f}"
                link_labels.append(link_label)

    # Create color scale based on fraction
    node_colors = []
    for node in filtered_data:
        fraction = node["fraction"]
        # Color gradient from light blue (low) to dark red (high)
        if fraction > 0.5:
            color = f"rgba(220, 20, 60, {0.3 + fraction * 0.7})"  # Crimson
        elif fraction > 0.1:
            color = f"rgba(255, 140, 0, {0.3 + fraction * 0.7})"  # Dark orange
        else:
            color = f"rgba(70, 130, 180, {0.3 + fraction * 0.7})"  # Steel blue
        node_colors.append(color)

    # Create link colors (lighter versions, with transparency)
    link_colors = []
    for target_idx in targets:
        node = filtered_data[target_idx]
        fraction = node["fraction"]
        if fraction > 0.5:
            color = "rgba(220, 20, 60, 0.2)"  # Light crimson
        elif fraction > 0.1:
            color = "rgba(255, 140, 0, 0.2)"  # Light orange
        else:
            color = "rgba(70, 130, 180, 0.2)"  # Light steel blue
        link_colors.append(color)

    # Create the Sankey diagram
    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=15,
                    thickness=20,
                    line=dict(color="black", width=0.5),
                    label=node_labels,
                    color=node_colors,
                    hovertemplate="%{label}<br>Score: %{value:.2f}<extra></extra>",
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color=link_colors,
                    hovertemplate="%{label}<extra></extra>",
                    customdata=link_labels,
                ),
                orientation="h",
                arrangement="snap",
            )
        ]
    )

    fig.update_layout(
        title=dict(
            text=title,
            font=dict(size=20, family="Arial, sans-serif"),
            x=0.5,
            xanchor="center",
        ),
        font=dict(size=12),
        height=height,
        width=width,
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    fig.show()


def _plot_sankey_matplotlib(
    data: List[Dict[str, Any]],
    title: str,
    value_field: str,
    height: int,
    width: int,
    max_label_length: int,
    min_fraction: float,
    show_values: bool,
) -> None:
    """
    Internal function to plot Sankey diagram using Matplotlib.

    Note: Matplotlib's Sankey is more limited and works best for simpler diagrams.
    For complex hierarchical trees, consider using Plotly instead.
    """

    # Filter data by minimum fraction
    filtered_data = [node for node in data if node.get("fraction", 0) >= min_fraction]

    if not filtered_data:
        print("No data to display after filtering.")
        return

    # Note: Matplotlib's Sankey is designed for flow diagrams, not hierarchical trees
    # This is a simplified implementation
    print("Warning: Matplotlib's Sankey has limited support for hierarchical trees.")
    print("For better results, use use_plotly=True")

    fig, ax = plt.subplots(figsize=(width / 100, height / 100))

    # Build tree level by level
    root = None
    for node in filtered_data:
        if node["parent"] is None:
            root = node
            break

    if root is None:
        print("No root node found.")
        return

    # Simple visualization: just show top-level contributions
    children = [node for node in filtered_data if node["parent"] == root["label"]]

    if not children:
        print("No children found for root node.")
        return

    # Sort children by fraction (descending)
    children.sort(key=lambda x: x["fraction"], reverse=True)

    # Create a simple Sankey diagram for one level
    sankey = Sankey(ax=ax, scale=0.01, offset=0.3, head_angle=120)

    # Calculate flows
    flows = []
    labels = []
    orientations = []

    # Input flow (root)
    flows.append(1.0)
    root_name = root["name"][:max_label_length]
    if show_values:
        labels.append(f"{root_name}\n{root['fraction'] * 100:.1f}%")
    else:
        labels.append(root_name)
    orientations.append(0)

    # Output flows (children)
    for child in children[:10]:  # Limit to top 10 for readability
        flows.append(-child["fraction"])
        child_name = child["name"][:max_label_length]
        if show_values:
            labels.append(f"{child_name}\n{child['fraction'] * 100:.1f}%")
        else:
            labels.append(child_name)
        orientations.append(1)

    # Add remaining as "others" if needed
    if len(children) > 10:
        remaining = sum(c["fraction"] for c in children[10:])
        flows.append(-remaining)
        labels.append(f"Others\n{remaining * 100:.1f}%")
        orientations.append(1)

    sankey.add(
        flows=flows,
        labels=labels,
        orientations=orientations,
        pathlengths=[0.25] * len(flows),
    )

    diagrams = sankey.finish()

    plt.title(title, fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()


def print_tree_structure(
    data: List[Dict[str, Any]],
    max_depth: Optional[int] = None,
    min_fraction: float = 0.0,
) -> None:
    """
    Print the tree structure in a readable text format.

    Parameters
    ----------
    data : List[Dict[str, Any]]
        List of dictionaries representing the tree nodes
    max_depth : int, optional
        Maximum depth to display (None for all levels)
    min_fraction : float, optional
        Minimum fraction to display (default: 0.0)
    """

    # Filter by minimum fraction
    filtered_data = [node for node in data if node.get("fraction", 0) >= min_fraction]

    # Build parent-to-children mapping
    children_map = {}
    root = None

    for node in filtered_data:
        parent = node.get("parent")
        if parent is None:
            root = node
        else:
            if parent not in children_map:
                children_map[parent] = []
            children_map[parent].append(node)

    # Sort children by fraction (descending)
    for children_list in children_map.values():
        children_list.sort(key=lambda x: x["fraction"], reverse=True)

    def print_node(node, depth=0, prefix=""):
        """Recursively print node and its children."""
        if max_depth is not None and depth > max_depth:
            return

        indent = "  " * depth
        fraction_pct = node["fraction"] * 100
        score = node["score"]
        name = node["name"]

        print(f"{indent}{prefix}{name}")
        print(f"{indent}  └─ Fraction: {fraction_pct:.2f}% | Score: {score:.2f}")

        # Print children
        label = node["label"]
        if label in children_map:
            for i, child in enumerate(children_map[label]):
                is_last = i == len(children_map[label]) - 1
                child_prefix = "└─ " if is_last else "├─ "
                print_node(child, depth + 1, child_prefix)

    if root:
        print("\n" + "=" * 80)
        print("LCA CONTRIBUTION TREE")
        print("=" * 80 + "\n")
        print_node(root)
        print("\n" + "=" * 80)
    else:
        print("No root node found in data.")

In [ ]:
plot_sankey_tree(calculations)